In [1]:
print("ok")

In [ ]:
import os
print(os.listdir("/lakehouse/default/Files"))

In [ ]:
import requests, hashlib, json, os
from datetime import datetime, timezone

# Pour une région

REGION = "ile-de-france"   # à adapter à votre zone
BASE = "https://download.geofabrik.de/europe/france"
url = f"{BASE}/{REGION}-latest.osm.pbf"

today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
dest_dir = f"/lakehouse/default/Files/bronze/osm/geofabrik/{REGION}/{today}"
os.makedirs(dest_dir, exist_ok=True)
dest = f"{dest_dir}/{REGION}-latest.osm.pbf"

# Téléchargement en streaming (le fichier ne passe jamais entièrement en mémoire)
md5 = hashlib.md5()
with requests.get(url, stream=True, timeout=60) as r:
    r.raise_for_status()
    last_modified = r.headers.get("Last-Modified")
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
            f.write(chunk)
            md5.update(chunk)

# Vérification d'intégrité avec le .md5 publié par Geofabrik
expected = requests.get(url + ".md5", timeout=30).text.split()[0]
assert md5.hexdigest() == expected, "MD5 différent : fichier corrompu, relancer"

# Manifeste de traçabilité (utile pour le livrable "liste des sources")
manifest = {
    "source": "OpenStreetMap via Geofabrik",
    "url": url,
    "region": REGION,
    "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_last_modified": last_modified,
    "size_bytes": os.path.getsize(dest),
    "md5": md5.hexdigest(),
    "license": "ODbL - © les contributeurs d'OpenStreetMap",
}
with open(f"{dest_dir}/_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(json.dumps(manifest, indent=2, ensure_ascii=False))

In [ ]:
import requests, hashlib, json, os
from datetime import datetime, timezone

BASE = "https://download.geofabrik.de/europe/france"

def ingest_osm(region: str) -> dict:
    # Télécharge l'extrait OSM Geofabrik d'une région dans le bronze, avec contrôle MD5 et manifeste.
    url = f"{BASE}/{region}-latest.osm.pbf"
    today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    dest_dir = f"/lakehouse/default/Files/bronze/osm/geofabrik/{region}/{today}"
    os.makedirs(dest_dir, exist_ok=True)
    dest = f"{dest_dir}/{region}-latest.osm.pbf"

    md5 = hashlib.md5()
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        last_modified = r.headers.get("Last-Modified")
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
                md5.update(chunk)

    expected = requests.get(url + ".md5", timeout=30).text.split()[0]
    if md5.hexdigest() != expected:
        raise ValueError(f"{region} : MD5 différent, fichier corrompu")

    manifest = {
        "source": "OpenStreetMap via Geofabrik",
        "url": url,
        "region": region,
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_last_modified": last_modified,
        "size_bytes": os.path.getsize(dest),
        "md5": md5.hexdigest(),
        "license": "ODbL - © les contributeurs d'OpenStreetMap",
    }
    with open(f"{dest_dir}/_manifest.json", "w") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    return manifest

REGIONS = [
    # Métropole (découpage Geofabrik = anciennes régions)
    "alsace", "aquitaine", "auvergne", "basse-normandie", "bourgogne",
    "bretagne", "centre", "champagne-ardenne", "corse", "franche-comte",
    "haute-normandie", "ile-de-france", "languedoc-roussillon", "limousin",
    "lorraine", "midi-pyrenees", "nord-pas-de-calais", "pays-de-la-loire",
    "picardie", "poitou-charentes", "provence-alpes-cote-d-azur", "rhone-alpes",
    # DROM
    "guadeloupe", "martinique", "guyane", "reunion", "mayotte",
]

ok, echecs = [], []
for region in REGIONS:
    try:
        print(f"→ {region}…", end=" ")
        m = ingest_osm(region)
        ok.append(region)
        print(f"OK ({m['size_bytes'] / 1e6:.0f} Mo)")
    except Exception as e:
        echecs.append(region)
        print(f"ÉCHEC : {e}")

print(f"\n{len(ok)}/{len(REGIONS)} régions ingérées.")
if echecs:
    print("À relancer :", echecs)